In [1]:
import json
import sys
import uuid
from pathlib import Path

import hiplot as hip
import numpy as np
import pandas as pd
from IPython.display import Javascript, display

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.plotting.reference_values import get_observer_reference_style
from tasks.policies import load_policy_results

zarr_path = project_root / "artifacts" / "simulation_runs" / "run-20260513T155848.zarr"
observable = "centered_mean_welfare.mean"
normalize_time = True


In [2]:
def filter_frame(frame, filters):
    """Return a filtered copy of ``frame`` while preserving all columns."""
    filtered = frame.copy()
    for column, raw_values in filters.items():
        if column not in filtered.columns:
            available = ", ".join(map(str, filtered.columns))
            raise KeyError(f"unknown filter column {column!r}; available columns: {available}")

        if isinstance(raw_values, (list, tuple, set, np.ndarray, pd.Index)) and not isinstance(raw_values, (str, bytes)):
            values = list(raw_values)
        else:
            values = [raw_values]

        if not values:
            raise ValueError(f"filter for {column!r} must contain at least one value")

        series = filtered[column]
        if pd.api.types.is_numeric_dtype(series) and all(
            isinstance(value, (int, float, np.integer, np.floating)) for value in values
        ):
            numeric = series.to_numpy(dtype=float)
            mask = np.zeros(len(filtered), dtype=bool)
            for value in values:
                if np.isnan(float(value)):
                    mask |= np.isnan(numeric)
                else:
                    mask |= np.isclose(numeric, float(value), rtol=0.0, atol=1e-9)
            filtered = filtered.loc[mask]
        else:
            filtered = filtered.loc[series.isin(values)]

    if filtered.empty:
        raise ValueError("reference_filter removed all rows")
    return filtered.reset_index(drop=True)


def _single_finite_value(frame, column):
    if column not in frame.columns:
        raise ValueError(
            f"target references require column {column!r}; keep it in parallel_frame "
            f"and set reference_filter = {{{column!r}: ...}} if needed"
        )

    values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]
    unique_values = np.unique(np.round(finite_values, 12))
    if len(unique_values) != 1:
        raise ValueError(
            f"target references require exactly one finite {column!r}; "
            f"found {len(unique_values)}. Set reference_filter = {{{column!r}: ...}} "
            "to choose one reference context, for example {'n_agents': 50}."
        )
    return float(unique_values[0])


def target_references_for_frame(frame, observable, theta_labels):
    """Resolve observer reference values for the displayed HiPlot frame."""
    if "target" not in frame.columns:
        raise ValueError("target_references_for_frame requires a 'target' column")

    missing_theta = [label for label in theta_labels if label not in frame.columns]
    if missing_theta:
        missing = ", ".join(missing_theta)
        raise ValueError(f"target_references_for_frame is missing theta columns: {missing}")

    n_agents = int(_single_finite_value(frame, "n_agents"))
    if observable == "centered_mean_welfare.mean":
        _single_finite_value(frame, "benefit_to_cost_ratio")
        _single_finite_value(frame, "distance_discount")

    references, _reference_styles = get_observer_reference_style(
        observable,
        frame["target"].to_numpy(dtype=float),
        n_agents=n_agents,
        theta_labels=theta_labels,
        theta_values=frame.loc[:, theta_labels].to_numpy(dtype=float),
    )
    return {
        str(label): float(value)
        for label, value in references.items()
        if np.isfinite(float(value))
    }


def display_hiplot_with_target_references(
    hip_exp,
    references,
    target_column="target",
    value_format="{:.2f}",
    right_padding=180,
):
    """Display a HiPlot experiment and overlay labeled reference arrows on target."""
    hip_display = hip_exp.display()
    reference_items = [
        {
            "label": str(label),
            "value": float(value),
            "text": f"{label} ({value_format.format(float(value))})",
        }
        for label, value in references.items()
        if np.isfinite(float(value))
    ]
    hip_exp._target_reference_overlay_js = ""
    if not reference_items:
        return hip_display

    overlay_class = f"hiplot-target-reference-overlay-{uuid.uuid4().hex}"
    marker_id = f"{overlay_class}-marker"
    overlay_js = f"""
(() => {{
  const references = {json.dumps(reference_items)};
  const targetColumn = {json.dumps(str(target_column))};
  const sharedOverlayClass = "hiplot-target-reference-overlay";
  const overlayClass = {json.dumps(overlay_class)};
  const markerId = {json.dumps(marker_id)};
  const rightPadding = Number({json.dumps(float(right_padding))});
  const ns = "http://www.w3.org/2000/svg";
  let animationFrame = null;
  let observer = null;

  function parseTranslate(transform) {{
    const match = /translate\\(\\s*([-+0-9.eE]+)(?:[,\\s]+([-+0-9.eE]+))?\\s*\\)/.exec(transform || "");
    if (!match) {{
      return [0, 0];
    }}
    return [Number(match[1] || 0), Number(match[2] || 0)];
  }}

  function makeSvg(tag, attrs) {{
    const node = document.createElementNS(ns, tag);
    for (const [key, value] of Object.entries(attrs || {{}})) {{
      node.setAttribute(key, value);
    }}
    return node;
  }}

  function observeRoot() {{
    if (!observer) {{
      return;
    }}
    document.querySelectorAll(".pplot-root").forEach((root) => {{
      observer.observe(root, {{ childList: true, subtree: true, attributes: true }});
    }});
  }}

  function ensureRightPadding(svg) {{
    const padding = Number.isFinite(rightPadding) ? Math.max(0, rightPadding) : 0;
    if (padding <= 0) {{
      return;
    }}
    if (!svg.dataset.targetReferenceBaseWidth) {{
      const attrWidth = Number(svg.getAttribute("width"));
      const rectWidth = svg.getBoundingClientRect().width;
      const baseWidth = Number.isFinite(attrWidth) && attrWidth > 0 ? attrWidth : rectWidth;
      svg.dataset.targetReferenceBaseWidth = String(baseWidth);
      const viewBox = svg.getAttribute("viewBox");
      if (viewBox) {{
        svg.dataset.targetReferenceBaseViewBox = viewBox;
      }}
    }}

    const baseWidth = Number(svg.dataset.targetReferenceBaseWidth);
    const paddedWidth = baseWidth + padding;
    svg.setAttribute("width", String(paddedWidth));
    svg.style.width = `${{paddedWidth}}px`;
    svg.style.maxWidth = "none";

    const baseViewBox = svg.dataset.targetReferenceBaseViewBox;
    if (baseViewBox) {{
      const parts = baseViewBox.trim().split(/[\\s,]+/).map(Number);
      if (parts.length === 4 && parts.every(Number.isFinite)) {{
        parts[2] = parts[2] + padding;
        svg.setAttribute("viewBox", parts.join(" "));
      }}
    }}

    const root = svg.closest(".pplot-root");
    if (root) {{
      root.style.paddingRight = `${{padding}}px`;
      root.style.boxSizing = "content-box";
    }}
    if (svg.parentElement) {{
      svg.parentElement.style.paddingRight = `${{padding}}px`;
      svg.parentElement.style.boxSizing = "content-box";
    }}
  }}

  function findTargetAxis(svg) {{
    return Array.from(svg.querySelectorAll("g.dimension")).find((axis) => {{
      const label = axis.querySelector(".label-name");
      return label && label.textContent.trim() === targetColumn;
    }});
  }}

  function axisTicks(axis) {{
    return Array.from(axis.querySelectorAll("g.tick"))
      .map((tick) => {{
        const text = tick.querySelector("text");
        if (!text) {{
          return null;
        }}
        const value = Number(text.textContent.trim().replace(/,/g, ""));
        const y = parseTranslate(tick.getAttribute("transform"))[1];
        if (!Number.isFinite(value) || !Number.isFinite(y)) {{
          return null;
        }}
        return {{ value, y }};
      }})
      .filter(Boolean);
  }}

  function interpolateY(value, ticks) {{
    if (ticks.length < 2) {{
      return null;
    }}
    const sorted = ticks.slice().sort((a, b) => a.value - b.value);
    const low = sorted[0];
    const high = sorted[sorted.length - 1];
    if (value < low.value || value > high.value || Math.abs(high.value - low.value) < 1e-12) {{
      return null;
    }}
    const ratio = (value - low.value) / (high.value - low.value);
    return low.y + ratio * (high.y - low.y);
  }}

  function clearOverlay(svg) {{
    svg.querySelectorAll(`.${{sharedOverlayClass}}`).forEach((node) => node.remove());
  }}

  function drawOverlay() {{
    if (observer) {{
      observer.disconnect();
    }}
    try {{
      const svgs = Array.from(document.querySelectorAll(".pplot-root svg"));
      const svg = svgs[svgs.length - 1];
      if (!svg) {{
        return false;
      }}

      clearOverlay(svg);
      const targetAxis = findTargetAxis(svg);
      if (!targetAxis) {{
        return false;
      }}

      const ticks = axisTicks(targetAxis);
      if (ticks.length < 2) {{
        return false;
      }}

      svg.style.overflow = "visible";
      const root = svg.closest(".pplot-root");
      if (root) {{
        root.style.overflow = "visible";
      }}
      if (svg.parentElement) {{
        svg.parentElement.style.overflow = "visible";
      }}
      ensureRightPadding(svg);

      const lineStart = 18;
      const lineEnd = 3;
      const textX = 22;
      const anchor = "start";

      const defs = makeSvg("defs", {{ class: `${{sharedOverlayClass}} ${{overlayClass}}` }});
      const marker = makeSvg("marker", {{
        id: markerId,
        markerWidth: "7",
        markerHeight: "7",
        refX: "6",
        refY: "3.5",
        orient: "auto",
        markerUnits: "strokeWidth",
        viewBox: "0 0 7 7",
      }});
      marker.appendChild(makeSvg("path", {{ d: "M 0 0 L 7 3.5 L 0 7 z", fill: "#c03728" }}));
      defs.appendChild(marker);
      svg.insertBefore(defs, svg.firstChild);

      const layer = makeSvg("g", {{ class: `${{sharedOverlayClass}} ${{overlayClass}}`, "pointer-events": "none" }});
      targetAxis.appendChild(layer);

      for (const item of references) {{
        const y = interpolateY(item.value, ticks);
        if (y === null) {{
          continue;
        }}
        const group = makeSvg("g", {{ class: "target-reference" }});
        group.appendChild(makeSvg("line", {{
          x1: String(lineStart),
          x2: String(lineEnd),
          y1: String(y),
          y2: String(y),
          stroke: "#c03728",
          "stroke-width": "1.6",
          "marker-end": `url(#${{markerId}})`,
        }}));

        const label = makeSvg("text", {{
          x: String(textX),
          y: String(y),
          dy: "0.32em",
          fill: "#c03728",
          "font-family": "sans-serif",
          "font-size": "11",
          "font-weight": "600",
          "paint-order": "stroke",
          stroke: "white",
          "stroke-width": "3",
          "stroke-linejoin": "round",
          "text-anchor": anchor,
        }});
        label.textContent = item.text;
        group.appendChild(label);
        layer.appendChild(group);
      }}
      return true;
    }} finally {{
      observeRoot();
    }}
  }}

  function scheduleDraw() {{
    if (animationFrame) {{
      window.cancelAnimationFrame(animationFrame);
    }}
    animationFrame = window.requestAnimationFrame(drawOverlay);
  }}

  let attempts = 0;
  const interval = window.setInterval(() => {{
    attempts += 1;
    if (drawOverlay() || attempts > 80) {{
      window.clearInterval(interval);
    }}
  }}, 100);

  observer = new MutationObserver(scheduleDraw);
  observeRoot();
  window.addEventListener("resize", scheduleDraw);
}})();
"""
    hip_exp._target_reference_overlay_js = overlay_js
    display(Javascript(overlay_js))
    return hip_display


In [3]:
ds = load_policy_results(zarr_path, normalize_time=normalize_time)
time_mode = "normalized" if normalize_time else "raw"
time_axis = ds.attrs.get("time_axis", "raw_simulator_steps")
final_step = float(ds.coords["step"].values[-1])

print("time mode:", time_mode)
print("time axis:", time_axis)
print("final step:", final_step)

theta_labels = [str(value) for value in ds.coords["theta_parameter"].values]
sweep_dims = tuple(dim for dim in ds["theta"].dims if dim != "theta_parameter")
sweep_columns = [
    dim
    for dim in sweep_dims
    if dim != "parameter_row" and dim not in theta_labels
]

theta = ds["theta"].transpose(*sweep_dims, "theta_parameter").to_numpy()
target = (
    ds["x"]
    .sel(observable=observable)
    .isel(step=-1)
    .transpose(*sweep_dims)
    .to_numpy()
)

mask = np.isfinite(theta).all(axis=-1) & np.isfinite(target)
theta_subset = theta[mask]
target_subset = target[mask]

parallel_frame = pd.DataFrame(theta_subset, columns=theta_labels)
for dim, values in zip(
    sweep_dims,
    np.meshgrid(
        *(ds.coords[dim].to_numpy() for dim in sweep_dims),
        indexing="ij",
    ),
    strict=True,
):
    if dim in sweep_columns:
        parallel_frame[dim] = values[mask]
parallel_frame["target"] = target_subset

dimensions = [*sweep_columns, *theta_labels, "target"]
parallel_frame = parallel_frame.loc[:, dimensions]

# Reference values are context-dependent. Keep the full zarr frame above, then
# choose one display/reference context here.
# Reference labels require a single context. Change this value to inspect a
# different n_agents slice while keeping parallel_frame fully dimensional.
reference_filter = {"n_agents": 50}
hiplot_frame = filter_frame(parallel_frame, reference_filter)

print("loaded:", zarr_path.relative_to(project_root))
print("rows:", len(parallel_frame))
print("display rows:", len(hiplot_frame))
print("reference filter:", reference_filter)
print("sweep dims:", list(sweep_dims))
print("dimensions:", dimensions)
hiplot_frame.head()


loaded: artifacts/simulation_runs/run-20260513T155848.zarr
rows: 40960
display rows: 8192
reference filter: {'n_agents': 50}
sweep dims: ['n_agents', 'best_response_proportion', 'parameter_row']
dimensions: ['best_response_proportion', 'n_agents', 'benefit_to_cost_ratio', 'distance_discount', 'visibility_distance', 'outside_optimism', 'exploration', 'target']


,best_response_proportion,n_agents,benefit_to_cost_ratio,distance_discount,visibility_distance,outside_optimism,exploration,target
0,1.0,50.0,0.1,0.9,3.0,0.564169,0.096206,0.192737
1,1.0,50.0,0.1,0.9,5.0,0.045771,0.023486,0.000000
2,1.0,50.0,0.1,0.9,1.0,0.366318,0.053020,-0.095842
3,1.0,50.0,0.1,0.9,3.0,0.773892,0.030440,0.517524
4,1.0,50.0,0.1,0.9,2.0,0.978768,0.067636,0.597765


In [4]:
hip_exp = hip.Experiment.from_dataframe(hiplot_frame)
hip_exp.colorby = "target"

target_references = target_references_for_frame(hiplot_frame, observable, theta_labels)
if target_references:
    target_range_values = np.array(
        [*hiplot_frame["target"].to_numpy(dtype=float), *target_references.values()],
        dtype=float,
    )
    target_range_values = target_range_values[np.isfinite(target_range_values)]
    hip_exp.parameters_definition["target"].force_range(
        float(target_range_values.min()),
        float(target_range_values.max()),
    )

display_hiplot_with_target_references(hip_exp, target_references)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
export_path = project_root / "artifacts" / "hiplot_with_references.html"
export_path.parent.mkdir(parents=True, exist_ok=True)

overlay_js = getattr(hip_exp, "_target_reference_overlay_js", "")
if target_references and not overlay_js:
    raise RuntimeError("Run the HiPlot display cell above before exporting so the reference overlay is generated.")

html = hip_exp.to_html()
if overlay_js:
    overlay_script = f"<script>{overlay_js}</script>"
    if "</body>" in html:
        html = html.replace("</body>", f"{overlay_script}</body>")
    else:
        html = f"{html}\n{overlay_script}"

export_path.write_text(html, encoding="utf-8")
export_path.relative_to(project_root)


PosixPath('artifacts/hiplot_with_references.html')